<a href="https://colab.research.google.com/github/Pranab9774/AI-ML/blob/main/DupeKiller.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq
!apt-get install -y openssh-client adb sshpass

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openssh-client is already the newest version (1:8.9p1-3ubuntu0.15).
The following additional packages will be installed:
  android-libadb android-libbase android-libboringssl android-libcrypto-utils
  android-libcutils android-liblog android-sdk-platform-tools-common
The following NEW packages will be installed:
  adb android-libadb android-libbase android-libboringssl
  android-libcrypto-utils android-libcutils android-liblog
  android-sdk-platform-tools-common sshpass
0 upgraded, 9 newly installed, 0 to remove and 9 not upgraded.
Need to get 998 kB of archives.
After this operation, 3,031 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 android-liblog

In [32]:
# ===== CHANGE THESE VALUES =====
PUBLIC_IP = "49.37.108.224"          # From whatismyip.com (e.g., 203.0.113.45)
SSH_USER = "sshuser"                   # Your SSH username
SSH_PASSWORD = "Joy@9774"         # Your SSH password
SSH_PORT = "22"                        # SSH port

# Example:
# PUBLIC_IP = "203.0.113.45"
# ================================

print("=" * 60)
print("CONNECTION DETAILS")
print("=" * 60)
print(f"Public IP:  {PUBLIC_IP}")
print(f"SSH User:   {SSH_USER}")
print(f"SSH Port:   {SSH_PORT}")
print("=" * 60)

CONNECTION DETAILS
Public IP:  49.37.108.224
SSH User:   sshuser
SSH Port:   22


In [33]:
import subprocess
import time

print("\n[1] Creating SSH tunnel...")
print("    Connecting Colab to your computer...")

tunnel_cmd = f"sshpass -p Joy@9774 ssh -N -L 5037:localhost:5037 sshuser@49.37.108.224 -p 22"

tunnel_process = subprocess.Popen(
    tunnel_cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("[2] Waiting for tunnel to establish...")
time.sleep(3)

if tunnel_process.poll() is None:
    print("✅ SSH TUNNEL ESTABLISHED!")
    print("   Colab is now connected to your computer")
else:
    stderr = tunnel_process.stderr.read().decode()
    print(f"❌ Tunnel failed: {stderr}")
    print("   Check your PUBLIC_IP and password")


[1] Creating SSH tunnel...
    Connecting Colab to your computer...
[2] Waiting for tunnel to establish...
✅ SSH TUNNEL ESTABLISHED!
   Colab is now connected to your computer


In [34]:
import time

print("\n[3] Restarting ADB daemon in Colab...")
!adb kill-server
time.sleep(2)
!adb start-server
time.sleep(2)

print("\n[4] Connecting to Android emulator via ADB...")
!adb connect localhost:5037
time.sleep(2)

print("\n[5] Listing connected devices...")
!adb devices

print("\n✅ ADB CONNECTION SUCCESSFUL!")


[3] Restarting ADB daemon in Colab...
* daemon not running; starting now at tcp:5037
* daemon started successfully

[4] Connecting to Android emulator via ADB...
failed to connect to localhost:5037

[5] Listing connected devices...
List of devices attached
localhost:5037	offline


✅ ADB CONNECTION SUCCESSFUL!


In [17]:
print("\n[5] Testing ADB shell...")
!adb shell whoami

print("\n[6] Getting Android info...")
!adb shell getprop ro.build.version.release
!adb shell getprop ro.product.model

print("\n✅ EVERYTHING IS WORKING!")


[5] Testing ADB shell...
error: device offline

[6] Getting Android info...
error: device offline
error: device offline

✅ EVERYTHING IS WORKING!


In [16]:
from google.colab import files
import zipfile

print("=" * 60)
print("UPLOADING DeepSeek_DupeKiller")
print("=" * 60)

print("\n📁 Click 'Choose Files' button that appears")
print("📂 Navigate to: D:\\AntiDiplo_Claude_version\\")
print("📦 Select: DeepSeek_DupeKiller.zip")
print("\nWaiting for your upload...\n")

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]
print(f"✅ Uploaded: {zip_file}")

print("\nExtracting ZIP file...")
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("✅ App extracted!")

print("\n" + "=" * 60)
print("YOUR APP FILES:")
print("=" * 60)
!ls -la /content/
!find /content -type f -name "*.py"

UPLOADING DeepSeek_DupeKiller

📁 Click 'Choose Files' button that appears
📂 Navigate to: D:\AntiDiplo_Claude_version\
📦 Select: DeepSeek_DupeKiller.zip

Waiting for your upload...



Saving DeepSeek_DupeKiller.zip to DeepSeek_DupeKiller.zip
✅ Uploaded: DeepSeek_DupeKiller.zip

Extracting ZIP file...
✅ App extracted!

YOUR APP FILES:
total 768
drwxr-xr-x 1 root root   4096 May 17 17:49 .
drwxr-xr-x 1 root root   4096 May 17 17:05 ..
drwxr-xr-x 4 root root   4096 May 12 13:29 .config
drwxr-xr-x 3 root root   4096 May 17 17:49 DeepSeek_DupeKiller
-rw-r--r-- 1 root root 762895 May 17 17:49 DeepSeek_DupeKiller.zip
drwxr-xr-x 1 root root   4096 May 12 13:29 sample_data
/content/DeepSeek_DupeKiller/Icons/Untitled-2.py
/content/DeepSeek_DupeKiller/Icons/from PIL import Image, ImageDraw, ImageF.py
/content/DeepSeek_DupeKiller/create_logo.py
/content/DeepSeek_DupeKiller/engine.py
/content/DeepSeek_DupeKiller/from kivy.py
/content/DeepSeek_DupeKiller/icon_manager.py
/content/DeepSeek_DupeKiller/test_kivy.py
/content/DeepSeek_DupeKiller/main.py


In [35]:
print("Enabling TCP mode...")
ssh_adb("tcpip 5555")

print("\nGetting emulator IP...")
stdout, stderr, code = ssh_adb("shell ip addr show wlan0 | grep inet | head -1 | awk '{print $2}' | cut -d/ -f1")
emulator_ip = stdout.strip()
print(f"Emulator: {emulator_ip}")

print("\nConnecting...")
ssh_adb(f"connect {emulator_ip}:5555")

print("\nPushing app...")
ssh_adb("push /data/local/tmp/DeepSeek_DupeKiller /data/local/tmp/")

print("\n✅ Running app...")
stdout, stderr, code = ssh_adb("shell 'cd /data/local/tmp/DeepSeek_DupeKiller && python3 main.py'")
print("Output:")
print(stdout)

Enabling TCP mode...

Getting emulator IP...
Emulator: 

Connecting...

Pushing app...

✅ Running app...
Output:

